# 🏠 London House Price Data — Data Cleaning Project

**Dataset:** London House Price Data (Kaggle) — 100,000 sold house prices from 1995 to Oct 2024  
**Source:** https://www.kaggle.com/datasets/jakewright/house-price-data  
**Author:** Alina Musteata  
**Date:** June 2026

---

## 🎯 Project Objectives

This notebook documents a full data cleaning pipeline on a large UK housing dataset. The cleaning steps include:

1. Initial data inspection
2. Handling missing values
3. Fixing data types
4. Standardising inconsistent formatting (postcodes, property types)
5. Detecting and handling outliers
6. Removing duplicates
7. Feature engineering
8. Exporting clean dataset

## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print('Libraries loaded successfully ✅')

## 2. 📂 Load the Dataset

In [ ]:
# Load raw data
df = pd.read_csv('../data/raw/kaggle_london_house_price_data.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

## 3. 🔍 Initial Data Inspection

In [ ]:
# Column data types
print('=== Data Types ===')
print(df.dtypes)
print()

# Basic stats
print('=== Basic Statistics ===')
df.describe(include='all')

In [ ]:
# Missing values audit
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('=== Missing Values Audit ===')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualise missing values
cols_with_missing = missing_df[missing_df['Missing Count'] > 0].index.tolist()

if cols_with_missing:
    plt.figure(figsize=(10, 5))
    missing_df[missing_df['Missing Count'] > 0]['Missing %'].plot(kind='bar', color='#E74C3C')
    plt.title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
    plt.xlabel('Column')
    plt.ylabel('Missing %')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../data/cleaned/missing_values_plot.png', dpi=150)
    plt.show()
else:
    print('No missing values found!')

## 4. 🧹 Step 1 — Handle Missing Values

In [ ]:
df_clean = df.copy()

# --- Numeric columns: fill with median ---
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'Filled "{col}" with median: {median_val:.2f}')

# --- Categorical columns: fill with mode or "Unknown" ---
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].isnull().mean() < 0.05:  # < 5% missing → fill with mode
            mode_val = df_clean[col].mode()[0]
            df_clean[col].fillna(mode_val, inplace=True)
            print(f'Filled "{col}" with mode: {mode_val}')
        else:  # >= 5% missing → fill with Unknown
            df_clean[col].fillna('Unknown', inplace=True)
            print(f'Filled "{col}" with "Unknown"')

print(f'\nRemaining nulls: {df_clean.isnull().sum().sum()}')

## 5. 🔧 Step 2 — Fix Data Types

In [ ]:
# Convert date column to datetime
date_cols = [col for col in df_clean.columns if 'date' in col.lower() or 'year' in col.lower()]

for col in date_cols:
    try:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        print(f'Converted "{col}" to datetime ✅')
    except Exception as e:
        print(f'Could not convert "{col}": {e}')

# Convert price column — strip £, commas, convert to numeric
price_cols = [col for col in df_clean.columns if 'price' in col.lower()]
for col in price_cols:
    if df_clean[col].dtype == 'object':
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.replace('[£,$,€,\,,\s]', '', regex=True)
            .str.strip()
        )
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f'Converted "{col}" to numeric ✅')

print('\nUpdated dtypes:')
print(df_clean.dtypes)

## 6. 📮 Step 3 — Standardise Postcode Formatting

In [ ]:
postcode_cols = [col for col in df_clean.columns if 'postcode' in col.lower() or 'post_code' in col.lower()]

for col in postcode_cols:
    # Uppercase, strip whitespace, normalise spacing
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)  # collapse multiple spaces
    )
    
    # Extract postcode district (e.g. "SW1A" from "SW1A 1AA")
    df_clean['postcode_district'] = df_clean[col].str.extract(r'^([A-Z]{1,2}\d{1,2}[A-Z]?)')
    
    print(f'Standardised "{col}" ✅')
    print(f'Sample: {df_clean[col].head(5).tolist()}')
    print(f'Unique districts: {df_clean["postcode_district"].nunique()}')

## 7. 🏷️ Step 4 — Standardise Property Type & Other Categoricals

In [ ]:
# Standardise property type values
property_col = [col for col in df_clean.columns if 'property' in col.lower() or 'type' in col.lower()]

for col in property_col:
    if df_clean[col].dtype == 'object':
        # Map common abbreviations → readable labels
        type_mapping = {
            'D': 'Detached',
            'S': 'Semi-Detached',
            'T': 'Terraced',
            'F': 'Flat/Maisonette',
            'O': 'Other',
            'detached': 'Detached',
            'semi-detached': 'Semi-Detached',
            'terraced': 'Terraced',
            'flat': 'Flat/Maisonette',
        }
        df_clean[col] = (
            df_clean[col]
            .str.strip()
            .str.title()
            .replace(type_mapping)
        )
        print(f'Property type value counts:\n{df_clean[col].value_counts()}')

# Standardise tenure (freehold/leasehold)
tenure_col = [col for col in df_clean.columns if 'tenure' in col.lower() or 'freehold' in col.lower()]
for col in tenure_col:
    if df_clean[col].dtype == 'object':
        tenure_mapping = {
            'F': 'Freehold', 'L': 'Leasehold', 'U': 'Unknown',
            'freehold': 'Freehold', 'leasehold': 'Leasehold'
        }
        df_clean[col] = df_clean[col].str.strip().str.title().replace(tenure_mapping)
        print(f'\nTenure value counts:\n{df_clean[col].value_counts()}')

## 8. 📊 Step 5 — Outlier Detection & Treatment

In [ ]:
# Focus on price column
price_col = [col for col in df_clean.columns if 'price' in col.lower()][0]

Q1 = df_clean[price_col].quantile(0.01)
Q3 = df_clean[price_col].quantile(0.99)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean[price_col] < lower_bound) | (df_clean[price_col] > upper_bound)]

print(f'Price column: "{price_col}"')
print(f'Lower bound: £{lower_bound:,.0f}')
print(f'Upper bound: £{upper_bound:,.0f}')
print(f'Outliers detected: {len(outliers):,} ({len(outliers)/len(df_clean)*100:.2f}%)')

# Visualise price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean[price_col].dropna(), bins=100, color='#3498DB', edgecolor='white')
axes[0].set_title('Price Distribution (Before Outlier Removal)', fontweight='bold')
axes[0].set_xlabel('Price (£)')
axes[0].set_ylabel('Frequency')

# Cap outliers using IQR method (winsorization)
df_clean[price_col + '_capped'] = df_clean[price_col].clip(lower=lower_bound, upper=upper_bound)

axes[1].hist(df_clean[price_col + '_capped'].dropna(), bins=100, color='#2ECC71', edgecolor='white')
axes[1].set_title('Price Distribution (After Capping)', fontweight='bold')
axes[1].set_xlabel('Price (£)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/cleaned/price_distribution.png', dpi=150)
plt.show()

## 9. 🔁 Step 6 — Remove Duplicates

In [ ]:
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
after = len(df_clean)

print(f'Rows before: {before:,}')
print(f'Rows after:  {after:,}')
print(f'Duplicates removed: {before - after:,}')

## 10. 🛠️ Step 7 — Feature Engineering

In [ ]:
# Extract year and month from date columns
date_cols = df_clean.select_dtypes(include=['datetime64']).columns.tolist()

for col in date_cols:
    df_clean[col + '_year'] = df_clean[col].dt.year
    df_clean[col + '_month'] = df_clean[col].dt.month
    df_clean[col + '_quarter'] = df_clean[col].dt.quarter
    print(f'Extracted year/month/quarter from "{col}" ✅')

# Price per decade bucket (for trend analysis)
if date_cols:
    year_col = date_cols[0] + '_year'
    df_clean['decade'] = (df_clean[year_col] // 10 * 10).astype(str) + 's'
    print(f'\nDecade distribution:\n{df_clean["decade"].value_counts().sort_index()}')

## 11. ✅ Final Validation & Summary

In [ ]:
print('=' * 50)
print('📋 CLEANING SUMMARY')
print('=' * 50)
print(f'Original shape:  {df.shape}')
print(f'Cleaned shape:   {df_clean.shape}')
print(f'Rows removed:    {df.shape[0] - df_clean.shape[0]:,}')
print(f'Columns added:   {df_clean.shape[1] - df.shape[1]}')
print(f'Remaining nulls: {df_clean.isnull().sum().sum()}')
print('=' * 50)
print()
print('Column overview:')
print(df_clean.dtypes)

## 12. 💾 Export Clean Dataset

In [ ]:
output_path = '../data/cleaned/london_house_prices_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f'Clean dataset saved to: {output_path} ✅')
print(f'Final shape: {df_clean.shape}')